# Lab 4: Preference-Based Post-Training -- DPO and GRPO

In this lab, you will implement two fundamental post-training algorithms from scratch using the Tinker training API:

1. **DPO (Direct Preference Optimization)**: Learn from human preference data to make a model more helpful and harmless
2. **GRPO (Group Relative Policy Optimization)**: Use verifiable rewards to improve a model's math reasoning

You will:
- Set up Tinker and understand its training primitives
- Fine-tune an SFT baseline model
- Implement the DPO loss function and train on preference pairs
- Implement GRPO with verifiable math rewards
- Compare all methods

**This is the solved version with all implementations filled in.**

---

## Section 0: Tinker Setup & Basics

Tinker is a cloud-based training API that lets you fine-tune large language models without needing a local GPU. In this section, you will install and configure Tinker, then verify that training and inference work correctly.

Nothing to implement here -- just run each cell and check the outputs.

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
SEED = 42
NUM_SFT_SAMPLES = 300
NUM_DPO_PAIRS = 300
NUM_GRPO_PROBLEMS = 200
DPO_BETA = 0.1
GRPO_GROUP_SIZE = 4
SFT_EPOCHS = 1
DPO_EPOCHS = 1
GRPO_ITERATIONS = 50
BATCH_SIZE = 4
LEARNING_RATE = 2e-4

print("Configuration loaded.")

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import torch
import json
import random
import math
import re
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import tinker
from tinker import types
from transformers import AutoTokenizer
from datasets import load_dataset
import warnings

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.style.use('seaborn-v0_8-whitegrid')
COLOR_SFT = '#4C72B0'
COLOR_DPO = '#DD8452'
COLOR_GRPO = '#55A868'

print("All imports loaded successfully.")

In [ ]:
# =============================================================================
# Connect to Tinker
# =============================================================================

service = tinker.ServiceClient()
capabilities = service.get_server_capabilities()

print("=" * 60)
print("Connected to Tinker!")
print("=" * 60)
print(f"\nSupported models:")
for model in capabilities.supported_models:
    print(f"  - {model.name}")
print()
print("Connection verified successfully.")

In [ ]:
# =============================================================================
# Load Tokenizer and Test Sampling
# =============================================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_sampling_client = service.create_sampling_client(model_path=MODEL_NAME)

test_messages = [{"role": "user", "content": "What is 2 + 2?"}]
test_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_tokens = tokenizer.encode(test_text)

test_input = types.ModelInput.from_ints(tokens=test_tokens)
test_params = types.SamplingParams(max_tokens=64, temperature=0.1)

test_result = base_sampling_client.sample(
    prompt=test_input, sampling_params=test_params, num_samples=1
).result()

print("Base model test response:")
print(tokenizer.decode(test_result.sequences[0].tokens, skip_special_tokens=True))
print("\nSampling client works!")

In [ ]:
# =============================================================================
# Test Training Client
# =============================================================================

test_training_client = service.create_lora_training_client(
    base_model=MODEL_NAME, rank=16
)

test_datum = types.Datum(
    model_input=types.ModelInput.from_ints(tokens=test_tokens[:50]),
    loss_fn_inputs=dict(
        target_tokens=test_tokens[1:51],
        weights=[1.0] * 50,
    )
)

test_fwd = test_training_client.forward([test_datum], "cross_entropy").result()
print(f"Forward pass test -- loss: {test_fwd.metrics.get('loss', 'N/A')}")
print(f"Logprobs shape: {len(test_fwd.loss_fn_outputs[0]['logprobs'].data)} values")
print("\nTraining client works! Tinker is fully set up.")

del test_training_client

---

## Section 1: SFT Baseline

Before we can do preference-based training, we need a supervised fine-tuned (SFT) model as our starting point. SFT trains the model with standard next-token prediction on high-quality demonstration data.

We will use a subset of the UltraFeedback dataset, taking only the "chosen" (preferred) responses as demonstrations.

In [ ]:
# =============================================================================
# Load SFT Training Data
# =============================================================================

raw_dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_sft")

sft_data = []
for i, sample in enumerate(raw_dataset):
    if i >= NUM_SFT_SAMPLES:
        break
    messages = sample["chosen"]
    if len(messages) >= 2 and messages[0]["role"] == "user" and messages[1]["role"] == "assistant":
        sft_data.append({
            "prompt": messages[0]["content"],
            "response": messages[1]["content"],
        })

print(f"Loaded {len(sft_data)} SFT training samples.")
print(f"\nExample prompt (first 200 chars):")
print(f"  {sft_data[0]['prompt'][:200]}...")
print(f"\nExample response (first 200 chars):")
print(f"  {sft_data[0]['response'][:200]}...")

In [ ]:
# =============================================================================
# PROVIDED: Prepare SFT Training Datums
# =============================================================================

def prepare_sft_datum(sample, tokenizer, max_length=512):
    """Convert a training sample into a Tinker Datum for SFT training."""
    messages = [
        {"role": "user", "content": sample["prompt"]},
        {"role": "assistant", "content": sample["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(text, truncation=True, max_length=max_length)
    target_tokens = tokens[1:] + [tokenizer.pad_token_id]
    
    prompt_messages = [{"role": "user", "content": sample["prompt"]}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_len = len(tokenizer.encode(prompt_text, truncation=True, max_length=max_length))
    
    weights = [0.0] * prompt_len + [1.0] * (len(tokens) - prompt_len)
    weights = weights[:len(tokens)]
    
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=tokens),
        loss_fn_inputs=dict(target_tokens=target_tokens, weights=weights),
    )

sft_datums = [prepare_sft_datum(s, tokenizer) for s in tqdm(sft_data, desc="Preparing SFT data")]
print(f"\nPrepared {len(sft_datums)} SFT datums.")
print(f"Example datum token length: {sft_datums[0].model_input.length}")

### 1.1 Train the SFT Model

**TODO**: Write the SFT training loop. Use `forward_backward()` with `"cross_entropy"` loss and `optim_step()` to train for `SFT_EPOCHS` epochs.

This is standard supervised fine-tuning -- iterate over batches of datums, compute the cross-entropy loss, and update the model weights.

In [ ]:
# =============================================================================
# SOLVED: Train the SFT Model
# =============================================================================

sft_training_client = service.create_lora_training_client(
    base_model=MODEL_NAME, rank=16
)

sft_losses = []

for epoch in range(SFT_EPOCHS):
    random.shuffle(sft_datums)
    epoch_losses = []
    
    for i in tqdm(range(0, len(sft_datums), BATCH_SIZE), desc=f"SFT Epoch {epoch+1}/{SFT_EPOCHS}"):
        batch = sft_datums[i:i+BATCH_SIZE]
        
        fwd = sft_training_client.forward_backward(batch, "cross_entropy")
        opt = sft_training_client.optim_step(types.AdamParams(learning_rate=LEARNING_RATE))
        
        result = fwd.result()
        opt.result()
        
        loss = result.metrics.get("loss", 0.0)
        epoch_losses.append(loss)
        sft_losses.append(loss)
    
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    print(f"  Epoch {epoch+1} -- avg loss: {avg_loss:.4f}")

print(f"\nSFT training complete. Final avg loss: {sft_losses[-1]:.4f}")

In [ ]:
# =============================================================================
# Save SFT Model and Test Generation
# =============================================================================

sft_sampling_client = sft_training_client.save_weights_and_get_sampling_client()
sft_training_client.save_state("sft-baseline-lab04").result()

def generate_response(sampling_client, tokenizer, prompt, max_tokens=256, temperature=0.7):
    """Generate a response from a sampling client."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    
    model_input = types.ModelInput.from_ints(tokens=tokens)
    params = types.SamplingParams(
        max_tokens=max_tokens,
        temperature=max(temperature, 0.01),
        stop=[tokenizer.eos_token] if tokenizer.eos_token else [],
    )
    
    result = sampling_client.sample(
        prompt=model_input, sampling_params=params, num_samples=1
    ).result()
    
    return tokenizer.decode(result.sequences[0].tokens, skip_special_tokens=True)

test_prompts = [
    "Explain the difference between a list and a tuple in Python.",
    "What are some healthy breakfast ideas?",
    "Write a haiku about machine learning.",
]

print("SFT Model Responses:")
print("=" * 60)
for prompt in test_prompts:
    response = generate_response(sft_sampling_client, tokenizer, prompt)
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response[:300]}...")
    print("-" * 60)

---

## Section 2: Direct Preference Optimization (DPO)

DPO is an elegant alternative to RLHF that directly optimizes a policy from preference data, without needing to train a separate reward model.

### The DPO Loss

Given a preference pair (prompt **x**, chosen response **y_w**, rejected response **y_l**), the DPO loss is:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\left(\beta \left[\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

Where:
- $\pi_\theta$ is the model being trained (initialized from SFT)
- $\pi_{\text{ref}}$ is the reference model (the frozen SFT model)
- $\beta$ controls the strength of the KL constraint
- $\sigma$ is the sigmoid function

**Intuition**: DPO increases the probability of chosen responses relative to rejected ones, but penalizes large deviations from the reference model (to prevent the model from degenerating).

In [ ]:
# =============================================================================
# Load Preference Data
# =============================================================================

pref_dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")

pref_data = []
for i, sample in enumerate(pref_dataset):
    if i >= NUM_DPO_PAIRS + 50:
        break
    chosen = sample["chosen"]
    rejected = sample["rejected"]
    
    if (len(chosen) >= 2 and len(rejected) >= 2 
        and chosen[0]["role"] == "user" and chosen[1]["role"] == "assistant"
        and rejected[0]["role"] == "user" and rejected[1]["role"] == "assistant"):
        pref_data.append({
            "prompt": chosen[0]["content"],
            "chosen": chosen[1]["content"],
            "rejected": rejected[1]["content"],
        })

dpo_train = pref_data[:NUM_DPO_PAIRS]
dpo_eval = pref_data[NUM_DPO_PAIRS:]

print(f"DPO training pairs: {len(dpo_train)}")
print(f"DPO eval pairs:     {len(dpo_eval)}")
print(f"\nExample preference pair:")
print(f"  Prompt:   {dpo_train[0]['prompt'][:150]}...")
print(f"  Chosen:   {dpo_train[0]['chosen'][:150]}...")
print(f"  Rejected: {dpo_train[0]['rejected'][:150]}...")

In [ ]:
# =============================================================================
# PROVIDED: Prepare DPO Datums
# =============================================================================

def prepare_dpo_datum(prompt, response, tokenizer, max_length=512):
    """Create a Datum for a prompt-response pair, suitable for computing log-probs."""
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(text, truncation=True, max_length=max_length)
    target_tokens = tokens[1:] + [tokenizer.pad_token_id]
    
    prompt_messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    prompt_len = len(tokenizer.encode(prompt_text, truncation=True, max_length=max_length))
    
    weights = [0.0] * prompt_len + [1.0] * (len(tokens) - prompt_len)
    weights = weights[:len(tokens)]
    
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=tokens),
        loss_fn_inputs=dict(target_tokens=target_tokens, weights=weights),
    )

print("DPO datum helper defined.")

### 2.1 Compute Reference Log-Probabilities

Before training, we need to compute the log-probabilities of both chosen and rejected responses under the **reference model** (our frozen SFT model). These reference log-probs stay fixed throughout DPO training.

**TODO**: Use the SFT training client's `forward()` method to compute per-token log-probs for each chosen and rejected response. Sum the response-token log-probs (using the weights mask) to get the total log-probability of each response.

In [ ]:
# =============================================================================
# SOLVED: Compute Reference Log-Probabilities
# =============================================================================

ref_logprobs_chosen = []
ref_logprobs_rejected = []

for pair in tqdm(dpo_train, desc="Computing reference logprobs"):
    # Chosen
    chosen_datum = prepare_dpo_datum(pair["prompt"], pair["chosen"], tokenizer)
    chosen_fwd = sft_training_client.forward([chosen_datum], "cross_entropy").result()
    chosen_lp = torch.tensor(chosen_fwd.loss_fn_outputs[0]["logprobs"].data)
    chosen_w = torch.tensor(chosen_datum.loss_fn_inputs["weights"].data)
    ref_logprobs_chosen.append((chosen_lp * chosen_w).sum().item())
    
    # Rejected
    rejected_datum = prepare_dpo_datum(pair["prompt"], pair["rejected"], tokenizer)
    rejected_fwd = sft_training_client.forward([rejected_datum], "cross_entropy").result()
    rejected_lp = torch.tensor(rejected_fwd.loss_fn_outputs[0]["logprobs"].data)
    rejected_w = torch.tensor(rejected_datum.loss_fn_inputs["weights"].data)
    ref_logprobs_rejected.append((rejected_lp * rejected_w).sum().item())

print(f"\nComputed reference logprobs for {len(ref_logprobs_chosen)} pairs.")
print(f"Mean ref logprob (chosen):   {np.mean(ref_logprobs_chosen):.2f}")
print(f"Mean ref logprob (rejected): {np.mean(ref_logprobs_rejected):.2f}")

### 2.2 Implement the DPO Loss Function

Now implement the core DPO loss. You will write a custom loss function that Tinker's `forward_backward_custom()` will call. This function receives the current model's per-token logprobs and must return a scalar loss.

The function signature is:
```python
def dpo_loss_fn(data, logprobs_list):
    # data: List[Datum] -- the datums you passed in (2 per pair: chosen + rejected)
    # logprobs_list: List[torch.Tensor] -- per-token logprobs from current model
    #   Each tensor has shape (seq_len,) and has requires_grad=True
    # Returns: (loss, metrics_dict)
```

**Key steps in your loss function:**
1. For each pair, extract the chosen and rejected logprobs tensors
2. Compute weighted sum of logprobs (multiply by weights, then sum) for chosen and rejected
3. Subtract the pre-computed reference logprobs to get log-ratios
4. Apply the DPO formula: `loss = -log(sigmoid(beta * (chosen_ratio - rejected_ratio)))`

In [ ]:
# =============================================================================
# SOLVED: DPO Loss Function
# =============================================================================

def make_dpo_loss_fn(batch_ref_chosen, batch_ref_rejected, beta=DPO_BETA):
    """Create a DPO loss function with captured reference logprobs.
    
    Args:
        batch_ref_chosen: List of reference logprobs for chosen responses in this batch
        batch_ref_rejected: List of reference logprobs for rejected responses in this batch
        beta: DPO temperature parameter
    
    Returns:
        A loss function compatible with forward_backward_custom()
    """
    def dpo_loss_fn(data, logprobs_list):
        total_loss = 0.0
        num_pairs = len(data) // 2
        
        chosen_rewards = []
        rejected_rewards = []
        
        for i in range(num_pairs):
            # Get current model logprobs (with gradients)
            chosen_logprobs = logprobs_list[2 * i]
            rejected_logprobs = logprobs_list[2 * i + 1]
            
            # Get weights to mask prompt tokens
            chosen_weights = torch.tensor(data[2 * i].loss_fn_inputs["weights"].data)
            rejected_weights = torch.tensor(data[2 * i + 1].loss_fn_inputs["weights"].data)
            
            # Compute weighted sum of logprobs (total log-prob of response)
            pi_chosen = (chosen_logprobs * chosen_weights).sum()
            pi_rejected = (rejected_logprobs * rejected_weights).sum()
            
            # Compute log-ratios against reference model
            chosen_ratio = pi_chosen - batch_ref_chosen[i]
            rejected_ratio = pi_rejected - batch_ref_rejected[i]
            
            # DPO loss for this pair
            loss_i = -torch.nn.functional.logsigmoid(
                beta * (chosen_ratio - rejected_ratio)
            )
            total_loss = total_loss + loss_i
            
            # Track implicit rewards for logging
            chosen_rewards.append(chosen_ratio.item())
            rejected_rewards.append(rejected_ratio.item())
        
        loss = total_loss / num_pairs
        metrics = {
            "dpo_loss": loss.item(),
            "chosen_reward": np.mean(chosen_rewards),
            "rejected_reward": np.mean(rejected_rewards),
            "reward_margin": np.mean(chosen_rewards) - np.mean(rejected_rewards),
        }
        return loss, metrics
    
    return dpo_loss_fn


print("DPO loss function defined.")

### 2.3 DPO Training Loop

Now put it all together. Create a new training client (initialized from the SFT weights), and train with your DPO loss function.

For each batch of preference pairs:
1. Prepare chosen and rejected datums (interleaved: chosen_0, rejected_0, chosen_1, rejected_1, ...)
2. Call `forward_backward_custom()` with your DPO loss function
3. Call `optim_step()` to update the model weights

In [ ]:
# =============================================================================
# SOLVED: DPO Training Loop
# =============================================================================

dpo_training_client = service.create_lora_training_client(
    base_model=MODEL_NAME, rank=16
)
dpo_training_client.load_state("sft-baseline-lab04").result()

dpo_losses = []

for epoch in range(DPO_EPOCHS):
    # Shuffle pairs and their reference logprobs together
    indices = list(range(len(dpo_train)))
    random.shuffle(indices)
    
    epoch_losses = []
    
    for batch_start in tqdm(range(0, len(indices), BATCH_SIZE), desc=f"DPO Epoch {epoch+1}/{DPO_EPOCHS}"):
        batch_indices = indices[batch_start:batch_start+BATCH_SIZE]
        
        # Prepare interleaved datums: [chosen_0, rejected_0, chosen_1, rejected_1, ...]
        batch_datums = []
        batch_ref_chosen = []
        batch_ref_rejected = []
        
        for idx in batch_indices:
            pair = dpo_train[idx]
            chosen_datum = prepare_dpo_datum(pair["prompt"], pair["chosen"], tokenizer)
            rejected_datum = prepare_dpo_datum(pair["prompt"], pair["rejected"], tokenizer)
            batch_datums.append(chosen_datum)
            batch_datums.append(rejected_datum)
            batch_ref_chosen.append(ref_logprobs_chosen[idx])
            batch_ref_rejected.append(ref_logprobs_rejected[idx])
        
        # Create loss function with this batch's reference logprobs
        loss_fn = make_dpo_loss_fn(batch_ref_chosen, batch_ref_rejected)
        
        # Forward-backward with custom DPO loss
        fwd = dpo_training_client.forward_backward_custom(batch_datums, loss_fn)
        opt = dpo_training_client.optim_step(types.AdamParams(learning_rate=LEARNING_RATE))
        
        result = fwd.result()
        opt.result()
        
        loss = result.metrics.get("dpo_loss", 0.0)
        epoch_losses.append(loss)
        dpo_losses.append(loss)
    
    avg_loss = sum(epoch_losses) / len(epoch_losses)
    print(f"  Epoch {epoch+1} -- avg DPO loss: {avg_loss:.4f}")

print(f"\nDPO training complete.")

In [ ]:
# =============================================================================
# Save DPO Model and Compare with SFT
# =============================================================================

dpo_sampling_client = dpo_training_client.save_weights_and_get_sampling_client()

comparison_prompts = [
    "How do I pick a lock?",
    "Write a persuasive essay about why homework should be banned.",
    "Explain quantum computing to a 5-year-old.",
    "What should I do if I'm feeling really stressed?",
]

print("SFT vs DPO Comparison:")
print("=" * 60)
for prompt in comparison_prompts:
    sft_response = generate_response(sft_sampling_client, tokenizer, prompt, max_tokens=200)
    dpo_response = generate_response(dpo_sampling_client, tokenizer, prompt, max_tokens=200)
    print(f"\nPrompt: {prompt}")
    print(f"\n  SFT: {sft_response[:250]}")
    print(f"\n  DPO: {dpo_response[:250]}")
    print("-" * 60)

### 2.4 Evaluate DPO

**TODO**: Evaluate the DPO model on the held-out eval pairs. For each pair, generate responses from both SFT and DPO models, then compute a simple preference metric: for how many prompts does the DPO response more closely match the "chosen" response (using length similarity and keyword overlap as proxies)?

In [ ]:
# =============================================================================
# SOLVED: Evaluate DPO vs SFT
# =============================================================================

eval_subset = dpo_eval[:20]  # Use 20 eval pairs

sft_wins = 0
dpo_wins = 0
ties = 0

for pair in tqdm(eval_subset, desc="Evaluating DPO"):
    sft_resp = generate_response(sft_sampling_client, tokenizer, pair["prompt"], max_tokens=200, temperature=0.3)
    dpo_resp = generate_response(dpo_sampling_client, tokenizer, pair["prompt"], max_tokens=200, temperature=0.3)
    
    chosen_words = set(pair["chosen"].lower().split())
    sft_overlap = len(set(sft_resp.lower().split()) & chosen_words)
    dpo_overlap = len(set(dpo_resp.lower().split()) & chosen_words)
    
    if dpo_overlap > sft_overlap:
        dpo_wins += 1
    elif sft_overlap > dpo_overlap:
        sft_wins += 1
    else:
        ties += 1

total = len(eval_subset)
print(f"\nDPO Evaluation Results (n={total}):")
print(f"  DPO wins:  {dpo_wins} ({dpo_wins/total:.1%})")
print(f"  SFT wins:  {sft_wins} ({sft_wins/total:.1%})")
print(f"  Ties:      {ties} ({ties/total:.1%})")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(["SFT", "DPO"], [sft_wins/total, dpo_wins/total], 
              color=[COLOR_SFT, COLOR_DPO], width=0.5)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f"{height:.1%}", ha='center', va='bottom', fontsize=12)
ax.set_ylabel("Win Rate vs Chosen Response", fontsize=12)
ax.set_title("DPO vs SFT: Alignment with Preferred Responses", fontsize=14)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

---

## Section 3: GRPO -- RL with Verifiable Rewards on Math

GRPO (Group Relative Policy Optimization) is a reinforcement learning method that does not require a learned reward model. Instead, it:

1. **Samples a group** of G responses for each prompt
2. **Scores each response** with a reward function (for math: is the answer correct?)
3. **Normalizes advantages** within the group: $A_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}$
4. **Updates the policy** using a weighted log-probability objective

This is particularly powerful for tasks with **verifiable rewards** (RLVR) -- tasks where we can automatically check if an answer is correct, like math problems.

### The GRPO Objective

For a prompt **x** and group of sampled responses $\{y_1, ..., y_G\}$ with rewards $\{r_1, ..., r_G\}$:

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G} A_i \cdot \log \pi_\theta(y_i | x)$$

where $A_i = \frac{r_i - \bar{r}}{\sigma_r + \epsilon}$ is the group-normalized advantage.

In [ ]:
# =============================================================================
# Load Math Data (GSM8K)
# =============================================================================

gsm_dataset = load_dataset("openai/gsm8k", "main", split="train")

math_data = []
for i, sample in enumerate(gsm_dataset):
    if i >= NUM_GRPO_PROBLEMS + 50:
        break
    answer_text = sample["answer"]
    match = re.search(r"####\s*(.+)", answer_text)
    if match:
        final_answer = match.group(1).strip().replace(",", "")
        math_data.append({
            "question": sample["question"],
            "answer": final_answer,
            "full_solution": answer_text,
        })

math_train = math_data[:NUM_GRPO_PROBLEMS]
math_eval = math_data[NUM_GRPO_PROBLEMS:]

print(f"GRPO training problems: {len(math_train)}")
print(f"GRPO eval problems:     {len(math_eval)}")
print(f"\nExample problem:")
print(f"  Question: {math_train[0]['question'][:200]}")
print(f"  Answer:   {math_train[0]['answer']}")

In [ ]:
# =============================================================================
# PROVIDED: Answer Extraction Helper
# =============================================================================

def extract_numerical_answer(text):
    """Extract the final numerical answer from a model's response."""
    match = re.search(r"####\s*([\d,]+\.?\d*)", text)
    if match:
        return match.group(1).replace(",", "")
    
    match = re.search(r"(?:the answer is|answer:|answer is)\s*([\d,]+\.?\d*)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "")
    
    match = re.search(r"=\s*([\d,]+\.?\d*)\s*$", text)
    if match:
        return match.group(1).replace(",", "")
    
    numbers = re.findall(r"[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "")
    
    return None

print("Answer extraction test:")
print(f"  'The answer is 42'  -> {extract_numerical_answer('The answer is 42')}")
print(f"  '#### 1,234'        -> {extract_numerical_answer('#### 1,234')}")
print(f"  'So 5 + 3 = 8'      -> {extract_numerical_answer('So 5 + 3 = 8')}")

### 3.1 Implement the Reward Function

**TODO**: Write a function that computes a binary reward: 1.0 if the model's extracted answer matches the ground truth, 0.0 otherwise.

In [ ]:
# =============================================================================
# SOLVED: Reward Function
# =============================================================================

def compute_reward(generated_text, correct_answer):
    """Compute binary reward: 1.0 if answer matches, 0.0 otherwise.
    
    Args:
        generated_text: The model's full response text
        correct_answer: The correct answer as a string
    
    Returns:
        float: 1.0 if correct, 0.0 if incorrect
    """
    extracted = extract_numerical_answer(generated_text)
    if extracted is None:
        return 0.0
    
    # Try exact string match
    if extracted.strip() == correct_answer.strip():
        return 1.0
    
    # Try numeric comparison
    try:
        if abs(float(extracted) - float(correct_answer)) < 1e-6:
            return 1.0
    except ValueError:
        pass
    
    return 0.0


# Test
print("Reward function tests:")
print(f"  compute_reward('The answer is 42', '42')     = {compute_reward('The answer is 42', '42')}")
print(f"  compute_reward('The answer is 43', '42')     = {compute_reward('The answer is 43', '42')}")
print(f"  compute_reward('I think it is about 5', '5') = {compute_reward('I think it is about 5', '5')}")
print(f"  compute_reward('No numbers here', '42')      = {compute_reward('No numbers here', '42')}")

### 3.2 Group Sampling and Advantage Computation

**TODO**: For each math problem, sample G responses from the current policy, compute rewards, and calculate group-normalized advantages.

The advantages tell the model which responses in the group were better than average (positive advantage) and which were worse (negative advantage).

In [ ]:
# =============================================================================
# SOLVED: Group Sampling and Advantage Computation
# =============================================================================

def sample_group_and_compute_advantages(sampling_client, tokenizer, problem, G=GRPO_GROUP_SIZE):
    """Sample G responses and compute group-normalized advantages.
    
    Args:
        sampling_client: Tinker SamplingClient
        tokenizer: HuggingFace tokenizer
        problem: Dict with 'question' and 'answer' keys
        G: Group size (number of samples per problem)
    
    Returns:
        list of dicts with keys: question, response, reward, advantage, tokens
    """
    prompt = (
        f"Solve this math problem step by step. "
        f"At the end, write your final answer after '####'.\n\n"
        f"{problem['question']}"
    )
    
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_tokens = tokenizer.encode(text)
    
    model_input = types.ModelInput.from_ints(tokens=prompt_tokens)
    params = types.SamplingParams(
        max_tokens=512,
        temperature=0.7,
        stop=[tokenizer.eos_token] if tokenizer.eos_token else [],
    )
    
    result = sampling_client.sample(
        prompt=model_input, sampling_params=params, num_samples=G
    ).result()
    
    group = []
    rewards = []
    
    for seq in result.sequences:
        response_text = tokenizer.decode(seq.tokens, skip_special_tokens=True)
        reward = compute_reward(response_text, problem["answer"])
        rewards.append(reward)
        
        # Get full conversation tokens for training
        full_messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response_text},
        ]
        full_text = tokenizer.apply_chat_template(full_messages, tokenize=False)
        full_tokens = tokenizer.encode(full_text, truncation=True, max_length=1024)
        
        group.append({
            "question": problem["question"],
            "response": response_text,
            "reward": reward,
            "tokens": full_tokens,
            "prompt_len": len(prompt_tokens),
        })
    
    # Compute group-normalized advantages
    mean_r = np.mean(rewards)
    std_r = np.std(rewards) + 1e-8
    
    for i, entry in enumerate(group):
        entry["advantage"] = (rewards[i] - mean_r) / std_r
    
    return group


# Test on one problem
test_group = sample_group_and_compute_advantages(
    base_sampling_client, tokenizer, math_train[0]
)
print(f"Group sampling test (G={GRPO_GROUP_SIZE}):")
for i, entry in enumerate(test_group):
    print(f"  Response {i+1}: reward={entry['reward']:.1f}, advantage={entry['advantage']:.2f}")
    print(f"    Answer: {extract_numerical_answer(entry['response'])}")
    print(f"    Correct: {math_train[0]['answer']}")

### 3.3 GRPO Training Loop

**TODO**: Implement the main GRPO training loop. For each iteration:
1. Sample a batch of problems
2. For each problem, sample a group of responses and compute advantages
3. Create datums from the sampled responses with advantages as weights
4. Use `forward_backward()` with the `"importance_sampling"` loss (which accepts per-token weights that encode the advantages)
5. Update model weights with `optim_step()`

The key insight: by setting the per-token weights to the advantages, the importance_sampling loss naturally implements the GRPO policy gradient -- responses with positive advantages get reinforced, while responses with negative advantages get suppressed.

In [ ]:
# =============================================================================
# SOLVED: GRPO Training Loop
# =============================================================================

grpo_training_client = service.create_lora_training_client(
    base_model=MODEL_NAME, rank=16
)
grpo_training_client.load_state("sft-baseline-lab04").result()

# Get initial sampling client
grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client()

grpo_rewards = []
grpo_accuracies = []

for iteration in tqdm(range(GRPO_ITERATIONS), desc="GRPO Training"):
    # Sample a random batch of problems
    batch_problems = random.sample(math_train, min(BATCH_SIZE, len(math_train)))
    
    # Collect datums and track rewards
    iter_datums = []
    iter_rewards = []
    
    for problem in batch_problems:
        group = sample_group_and_compute_advantages(
            grpo_sampling_client, tokenizer, problem
        )
        
        for entry in group:
            iter_rewards.append(entry["reward"])
            
            # Skip responses with zero advantage (no learning signal)
            if abs(entry["advantage"]) < 1e-8:
                continue
            
            tokens = entry["tokens"]
            target_tokens = tokens[1:] + [tokenizer.pad_token_id]
            
            # Set weights: 0 on prompt, advantage on response
            prompt_len = entry["prompt_len"]
            weights = ([0.0] * prompt_len + 
                      [entry["advantage"]] * (len(tokens) - prompt_len))
            weights = weights[:len(tokens)]
            
            datum = types.Datum(
                model_input=types.ModelInput.from_ints(tokens=tokens),
                loss_fn_inputs=dict(
                    target_tokens=target_tokens,
                    weights=weights,
                ),
            )
            iter_datums.append(datum)
    
    # Update model if we have any datums with non-zero advantages
    if iter_datums:
        fwd = grpo_training_client.forward_backward(iter_datums, "importance_sampling")
        opt = grpo_training_client.optim_step(types.AdamParams(learning_rate=LEARNING_RATE * 0.5))
        fwd.result()
        opt.result()
    
    mean_reward = np.mean(iter_rewards) if iter_rewards else 0.0
    grpo_rewards.append(mean_reward)
    
    # Update sampling client periodically
    if (iteration + 1) % 10 == 0:
        grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client()
        print(f"  Iter {iteration+1}/{GRPO_ITERATIONS} -- mean reward: {mean_reward:.3f}")

# Final sampling client update
grpo_sampling_client = grpo_training_client.save_weights_and_get_sampling_client()
print(f"\nGRPO training complete. Final mean reward: {grpo_rewards[-1]:.3f}")

### 3.4 Evaluate GRPO Model

**TODO**: Evaluate the GRPO model on held-out math problems. Compare accuracy against the base model and SFT model.

In [ ]:
# =============================================================================
# SOLVED: Evaluate GRPO
# =============================================================================

def evaluate_math_model(sampling_client, tokenizer, eval_set, num_samples=None):
    """Evaluate a model on math problems.
    
    Returns accuracy (fraction of problems solved correctly).
    """
    samples = eval_set[:num_samples] if num_samples else eval_set
    correct = 0
    
    for problem in tqdm(samples, desc="Evaluating"):
        prompt = (
            f"Solve this math problem step by step. "
            f"At the end, write your final answer after '####'.\n\n"
            f"{problem['question']}"
        )
        response = generate_response(
            sampling_client, tokenizer, prompt, max_tokens=512, temperature=0.1
        )
        reward = compute_reward(response, problem["answer"])
        correct += reward
    
    accuracy = correct / len(samples)
    return accuracy


eval_subset = math_eval[:30]  # Evaluate on 30 problems

print("Evaluating base model on math...")
base_math_acc = evaluate_math_model(base_sampling_client, tokenizer, eval_subset)
print(f"  Base model accuracy: {base_math_acc:.1%}")

print("\nEvaluating SFT model on math...")
sft_math_acc = evaluate_math_model(sft_sampling_client, tokenizer, eval_subset)
print(f"  SFT model accuracy: {sft_math_acc:.1%}")

print("\nEvaluating GRPO model on math...")
grpo_math_acc = evaluate_math_model(grpo_sampling_client, tokenizer, eval_subset)
print(f"  GRPO model accuracy: {grpo_math_acc:.1%}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
models = ["Base", "SFT", "GRPO"]
accs = [base_math_acc, sft_math_acc, grpo_math_acc]
colors = ["#808080", COLOR_SFT, COLOR_GRPO]
bars = ax.bar(models, accs, color=colors, width=0.5)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f"{height:.1%}", ha='center', va='bottom', fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Math Problem Accuracy: Base vs SFT vs GRPO", fontsize=14)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

---

## Section 4: Comparison & Analysis

Now let's compare all the models and training methods we have explored.

### 4.1 Training Curves

**TODO**: Plot the training curves for SFT, DPO, and GRPO side by side. This helps visualize how each method converges.

In [ ]:
# =============================================================================
# SOLVED: Training Curves
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# SFT loss
axes[0].plot(sft_losses, color=COLOR_SFT, linewidth=1.5)
axes[0].set_title("SFT: Cross-Entropy Loss", fontsize=14)
axes[0].set_xlabel("Step", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].grid(True, alpha=0.3)

# DPO loss
axes[1].plot(dpo_losses, color=COLOR_DPO, linewidth=1.5)
axes[1].set_title("DPO: Preference Loss", fontsize=14)
axes[1].set_xlabel("Step", fontsize=12)
axes[1].set_ylabel("Loss", fontsize=12)
axes[1].grid(True, alpha=0.3)

# GRPO reward
axes[2].plot(grpo_rewards, color=COLOR_GRPO, linewidth=1.5)
axes[2].set_title("GRPO: Mean Reward", fontsize=14)
axes[2].set_xlabel("Iteration", fontsize=12)
axes[2].set_ylabel("Mean Reward", fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 Side-by-Side Model Outputs

**TODO**: Generate responses from all models on the same set of prompts to qualitatively compare their behaviors.

In [ ]:
# =============================================================================
# SOLVED: Side-by-Side Model Comparison
# =============================================================================

comparison_prompts = [
    "How do I make a friend feel better after a bad day?",
    "What is the square root of 144?",
    "Should I lie on my resume to get a better job?",
    "Solve: If a train travels 60 mph for 2.5 hours, how far does it go?",
]

models = {
    "Base": base_sampling_client,
    "SFT": sft_sampling_client,
    "DPO": dpo_sampling_client,
    "GRPO": grpo_sampling_client,
}

for prompt in comparison_prompts:
    print("=" * 70)
    print(f"PROMPT: {prompt}")
    print("=" * 70)
    for name, client in models.items():
        response = generate_response(client, tokenizer, prompt, max_tokens=200, temperature=0.3)
        print(f"\n  [{name}]: {response[:300]}")
    print()

### 4.3 Summary Chart

**TODO**: Create a summary visualization showing what each method is good at.

In [ ]:
# =============================================================================
# SOLVED: Summary Comparison Chart
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

models_list = ["Base", "SFT", "DPO", "GRPO"]
math_accs = [base_math_acc, sft_math_acc, sft_math_acc, grpo_math_acc]  # DPO not trained on math
colors = ["#808080", COLOR_SFT, COLOR_DPO, COLOR_GRPO]

x = np.arange(len(models_list))
width = 0.5

bars = ax.bar(x, math_accs, width, color=colors)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f"{height:.1%}", ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("Math Accuracy", fontsize=12)
ax.set_title("Post-Training Method Comparison: Math Accuracy", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(models_list)
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

print("\nKey Takeaways:")
print("  - SFT provides a solid baseline by learning from demonstrations")
print("  - DPO optimizes for human preferences (helpfulness/safety) but may not improve math")
print("  - GRPO with verifiable rewards directly improves math reasoning")
print("  - Different post-training methods target different objectives")

---

## Section 5: Reflection

Take a few minutes to think about the following questions. There are no "right" answers -- the goal is to build intuition about when and why to use different post-training methods.

### 5.1 DPO vs GRPO

When would you choose DPO over GRPO, and vice versa? Consider:
- What kind of data does each method require?
- What tasks is each method best suited for?
- What are the computational tradeoffs?

*Write your response below:*

*Your answer here...*

### 5.2 The Role of the Reference Model

Both DPO and GRPO use a reference model (the SFT model). Why is this important? What would happen if we did not include the KL constraint against the reference model?

*Write your response below:*

*Your answer here...*

### 5.3 Verifiable vs Non-Verifiable Rewards

GRPO works well for math because we can automatically verify answers. But many important tasks (writing quality, helpfulness, safety) do not have easily verifiable rewards.

- How could you adapt GRPO for tasks without verifiable rewards?
- What are the risks of using a learned reward model instead of verifiable rewards?
- How does this connect to the broader challenge of aligning language models?

*Write your response below:*

*Your answer here...*

### 5.4 Scaling and Practical Considerations

In this lab, you trained on small datasets with a single model. In practice, companies train on millions of preference pairs with much larger models.

- How do you think the effectiveness of DPO vs GRPO changes with scale?
- What practical challenges would you expect when scaling these methods?
- What role does data quality play in preference-based training?

*Write your response below:*

*Your answer here...*

---

## Congratulations!

You have completed the RLHF lab. Here is a summary of what you accomplished:

1. **Set up Tinker** and learned to use its training primitives (forward, backward, optimize, sample)
2. **Trained an SFT baseline** using standard next-token prediction on demonstration data
3. **Implemented DPO from scratch** -- writing the loss function and training loop to learn from human preferences
4. **Implemented GRPO from scratch** -- using verifiable math rewards to improve reasoning without human labels
5. **Compared all methods** and understood when to use each approach

These are the core algorithms behind how modern language models like Claude, ChatGPT, and Gemini are trained after pre-training. Understanding them at this level gives you deep insight into how AI alignment works in practice.